# Gold — fact_sales
**GlobalMart Orchestration Lab**

| | |
|---|---|
| **Grain** | One row per transaction |
| **Sources** | `silver.transactions` (drives the grain) + `silver.orders` + `silver.customers` |
| **Target** | `{catalog}.gold.fact_sales` |
| **Pattern** | Join chain -> `sha2()` surrogate key -> MERGE, same shape as `Databricks/Day7/Day7_3_HOL2_Build_Gold_Layer_Fact_Sales.ipynb` |
| **Depends on** | Silver Customers, Silver Orders, Silver Transactions — all three must complete first |

MERGE instead of Day 7's `overwrite`, because this lab needs to support the incremental second run without re-processing the whole table.

## Step 1 — Setup

In [ ]:
from pyspark.sql.functions import col, sha2, current_timestamp
from delta.tables import DeltaTable

dbutils.widgets.text('catalog',       'your_catalog')
dbutils.widgets.text('source_schema', 'silver')
dbutils.widgets.text('target_schema', 'gold')

CATALOG       = dbutils.widgets.get('catalog')
SOURCE_SCHEMA = dbutils.widgets.get('source_schema')
TARGET_SCHEMA = dbutils.widgets.get('target_schema')

SILVER_TXN       = f'{CATALOG}.{SOURCE_SCHEMA}.transactions'
SILVER_ORDERS    = f'{CATALOG}.{SOURCE_SCHEMA}.orders'
SILVER_CUSTOMERS = f'{CATALOG}.{SOURCE_SCHEMA}.customers'
TABLE            = f'{CATALOG}.{TARGET_SCHEMA}.fact_sales'

print(f'Target: {TABLE}')

## Step 2 — Read Silver Tables
Transactions drive the grain — one fact row per transaction. Orders and customers are joined in to enrich each row with context.

In [ ]:
txn    = spark.table(SILVER_TXN).alias("txn")
ord_df = spark.table(SILVER_ORDERS).alias("ord")
cust   = spark.table(SILVER_CUSTOMERS).alias("cust")

print(f"silver.transactions rows : {txn.count():,}")
print(f"silver.orders rows       : {ord_df.count():,}")
print(f"silver.customers rows    : {cust.count():,}")

## Step 3 — Build Fact Rows

Join chain:
1. `transactions` → `orders` on `order_id` (**inner** — every transaction must have a real order; Silver already quarantined anything that didn't)
2. `orders` → `customers` on `customer_id` (**left** — defensive re-check, not a source of columns. Silver orders already verified this `customer_id` existed in `silver.customers` at ingestion time; this join catches the rare case where that customer was since removed, without dropping the transaction.)

In [ ]:
fact_df = (
    txn
    .join(ord_df, col("txn.order_id") == col("ord.order_id"), "inner")
    .join(cust,   col("ord.customer_id") == col("cust.customer_id"), "left")
    .select(
        col("txn.transaction_id"),
        col("txn.order_id"),
        col("ord.customer_id"),
        col("ord.order_date"),
        col("ord.status"),
        col("txn.payment_mode"),
        col("txn.quantity"),
        col("txn.unit_price"),
        col("txn.total_amount"),
        col("cust.customer_id").alias("_customer_verified"),
    )
)
print(f"Rows after join chain: {fact_df.count():,}")

# Sanity check -- tests the customers LEFT JOIN actually matched, not ord.customer_id
# (which is never null here since Silver orders already quarantines those).
# Silver orders enforces this RI check at ingestion time, so this should be 0;
# non-zero would mean a customer was deleted from silver.customers afterward.
missing_customer = fact_df.filter(col("_customer_verified").isNull()).count()
print(f"Orders whose customer no longer exists in silver.customers: {missing_customer:,}  (expected 0)")

fact_df = fact_df.drop("_customer_verified")

## Step 4 — Add Surrogate Key
`fact_sales_sk = sha2(transaction_id, 256)` — same pattern as Day 7's `fact_sales_sk = sha2(order_item_id, 256)`. `transaction_id` is already the natural grain key, so hashing it gives a stable, deterministic surrogate key without inventing a new identity.

In [ ]:
fact_df = fact_df.withColumn("fact_sales_sk", sha2(col("transaction_id"), 256)) \
    .select(
        "fact_sales_sk", "transaction_id", "order_id", "customer_id",
        "order_date", "status", "payment_mode",
        "quantity", "unit_price", "total_amount"
    )

print(f"Fact rows ready to MERGE: {fact_df.count():,}")
fact_df.display()

## Step 5 — Create Gold Table (first run only)

In [ ]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{TARGET_SCHEMA}")

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {TABLE} (
        fact_sales_sk  STRING,
        transaction_id STRING,
        order_id       STRING,
        customer_id    STRING,
        order_date     DATE,
        status         STRING,
        payment_mode   STRING,
        quantity       INT,
        unit_price     DOUBLE,
        total_amount   DOUBLE
    )
    USING DELTA
""")
print(f"Table ready: {TABLE}")

## Step 6 — MERGE into fact_sales
Match on `fact_sales_sk` — update all columns if the transaction was updated upstream (a status change flows through from `silver.orders`), insert if new. Same code, unchanged, for batch 1 and batch 2.

In [ ]:
target = DeltaTable.forName(spark, TABLE)

(target.alias("t")
    .merge(fact_df.alias("s"), "t.fact_sales_sk = s.fact_sales_sk")
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)
print("MERGE complete")

## Step 7 — Verify

In [ ]:
from pyspark.sql.functions import sum as spark_sum, round as spark_round, count as spark_count

result = spark.table(TABLE)
print(f"Total rows in {TABLE}: {result.count():,}")

print("\n--- Revenue by Status ---")
result.groupBy("status").agg(
    spark_count("*").alias("transactions"),
    spark_round(spark_sum("total_amount"), 2).alias("total_revenue")
).display()

In [ ]:
# Delta history -- confirms this run actually wrote a new MERGE version
spark.sql(f"DESCRIBE HISTORY {TABLE}") \
    .select("version", "timestamp", "operation", "operationMetrics") \
    .orderBy(col("version").desc()).limit(4).display()